# KALIA X20 — was it ever a gate?

X19 removed the gate from Gated Residual and the gain vanished. But G-0 had already
measured that gate as non-responsive to its input: a standard deviation of 5e-06
across documents against a mean of 0.019. Both measurements are correct, and they
do not sit together. A component cannot be irrelevant to its input and
load-bearing at the same time unless it is doing something other than what it is
named.

The candidate reading: with `g ≈ 0.019` the module is a **learned, static,
per-channel modulation** of the normalised stream — a rank-32 low-rank adapter on
the residual path, wearing the name of a data-dependent gate.

So this runs the same arm with the input path deleted: `w1`'s weight becomes a
learned constant, everything else is identical, and the module is **73,728
parameters smaller** than the gated one. If the smaller arm matches the larger,
the input path is not where the work is.

| arm | params | val loss |
|---|---|---|
| control | 29,920,512 | 4.7662 |
| branchnorm (X19) | 29,922,816 | 4.7680 |
| gated (X18) | 30,072,768 | 4.7226 |
| **staticgate** | **29,999,040** | *this run* |

**H-1 bar: 4.7544** — X19's G-1 bar, reused rather than reinvented, so the two
experiments stay directly comparable.

The prior is recorded against my own convenience: a null here is the more
interesting result, because it would mean the gate does need its input and G-0 was
simply wrong.

In [ ]:
import glob, os, shutil, subprocess, sys

work = '/kaggle/working/kalia'
if not os.path.exists(work):
    hits = sorted(glob.glob("/kaggle/input/**/ablate.py", recursive=True))
    assert hits, "attach the kalia-code-dev dataset (must be current: it carries configs/ and eval/)"
    shutil.copytree(os.path.dirname(hits[0]), work)
os.chdir(work)
print('code from', work)
print('branch_norm in model.py:', 'class BranchNorm' in open('model.py').read())
data = sorted(glob.glob('/kaggle/input/**/train.bin', recursive=True))
assert data, 'train.bin not found - attach the kalia-prep-v2b output'
print('data:', data[0])
cfg = open('configs/micro-staticgate.yaml').read() if os.path.exists('configs/micro-staticgate.yaml') else ''
print('arm config present:', bool(cfg))

In [ ]:
print(subprocess.run([sys.executable, 'ablate.py',
                       '--arms', 'micro-staticgate',
                       '--steps', '500',
                       '--data-dir', str(os.path.dirname(data[0])),
                       '--out-root', '/kaggle/working/out'],
                      capture_output=True, text=True).stdout[-4000:])

In [ ]:
import pandas as pd
p = '/kaggle/working/out/micro-staticgate/val_log.csv'
df = pd.read_csv(p)
print('micro-staticgate: ' + '  '.join(f"{int(r.step)}={r.val_loss:.4f}" for r in df.itertuples()))
loss = float(df.iloc[-1].val_loss)
print(f'\nfinal val loss {loss:.4f}')

In [ ]:
# G-2 re-asserted on the checkpoint that actually trained, not on a
# freshly constructed model. Amendment 1 corrected the expected count from
# 29,920,512 to 29,922,816 (RMSNorm is affine, so the branch norm is not
# free) and the gate's own share to 149,952.
import torch
ck = torch.load('/kaggle/working/out/micro-staticgate/ckpt.pt',
                 map_location='cpu', weights_only=False)
sd = ck['model']
# Count *parameters*, not state-dict entries. lm_head is tied to tok_emb, so
# summing sd.values() counts the embedding twice: 29,922,816 + 19,298,688 =
# 49,221,504, which is what version 1 of this kernel reported and called a
# FAIL. The model was correct; the check was not.
n_params = sum(v.numel() for k, v in sd.items() if k != 'lm_head.weight')
EXPECTED = 29999040
print(f'G-2: checkpoint params {n_params:,} | expected {EXPECTED:,} | '
      + ('PASS' if n_params == EXPECTED else 'FAIL'))
gates = [k for k in sd if '.gated.' in k]
branches = [k for k in sd if '.branch.' in k]
print(f'G-2: gate tensors present: {len(gates)} (expected 0 for this arm)')
print(f'G-2: branch tensors present: {len(branches)}')
assert not gates, 'this arm must carry no gate parameters'
assert n_params == EXPECTED, f'parameter count {n_params} != {EXPECTED}'
print('G-2 PASS')

In [ ]:
CONTROL, GATED, BRANCHNORM, BAR = 4.7662, 4.7226, 4.768, 4.7544
print('=== X19 G-1 ===')
print(f'  control      {CONTROL:.4f}')
print(f'  gated        {GATED:.4f}   ({GATED-CONTROL:+.4f} vs control)')
print(f'  branchnorm   {BRANCHNORM:.4f}   ({BRANCHNORM-CONTROL:+.4f} vs control)')
print(f'  staticgate   {loss:.4f}   ({loss-CONTROL:+.4f} vs control, {loss-GATED:+.4f} vs gated)')
print(f'  G-1 bar      {BAR:.4f}   (midpoint + 0.010)')
print()
if loss <= BAR:
    print('  H-1 PASS  the static adapter reproduces the gain')
    print('  => the data-dependent read contributes nothing. The gate was')
    print('     never a gate: the name transferred, the behaviour did not.')
    print('  => 73,728 fewer parameters for the same result.')
else:
    print(f'  H-1 FAIL  {loss-BAR:+.4f} outside the bar')
    print('  => the gate DOES need its input, so G-0 measured it wrong,')
    print('     and the mechanism is real — just not as documented.')
    print('  => a cheaper diagnostic (mean and dispersion) can falsify a')
    print('     borrowed claim, which is itself worth publishing.')
print()
print('Caveat carried from X18 and still binding here: single seed, 30M '
      'parameters, 500 steps. A screening window, not a training run.')

Read alongside X18, not on its own.

- **G-1 pass** → X18's gain is attributed: branch normalisation, gate inert.
  A free architectural change and a 149,952-parameter deletion.
- **G-1 fail** → the branch structure does not explain the gain. Combined with
  G-0's zero input-dependence, that leaves X18's −0.0436 most likely a
  single-seed artifact, which would be the honest conclusion and would mean
  the gate *and* the branch norm both go.

Either way the registered thresholds are reported as written. G-1 is a
loss threshold in nats, so unlike X18's accuracy bar it is not sitting below
the measurement's own noise (see D45).